In [1]:
import pandas as pd

# ---- 1. Load both prediction files ----
rule_df = pd.read_csv('../data/processed/amazon_brand_mislabel_pairs.csv')  # has rule preds only if you saved them; re-derive if not
llm_df = pd.read_csv('../data/processed/amazon_brand_llm_zeroshot_preds.csv')

In [2]:
# If rule predictions weren't saved separately on Day 11, recompute them here quickly:
import re, string

def normalize(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def rule_based_check(title, manufacturer_shown):
    title_norm = normalize(title)
    manu_norm = normalize(manufacturer_shown)
    return 1 if manu_norm in title_norm else 0  # 1 = looks correct

rule_df['rule_pred_correct'] = rule_df.apply(
    lambda row: rule_based_check(row['title'], row['manufacturer_shown']), axis=1
)
rule_df['rule_pred_mislabeled'] = 1 - rule_df['rule_pred_correct']

In [3]:
# ---- 2. Merge on id ----
merged = rule_df[['id', 'title', 'manufacturer_shown', 'true_manufacturer', 'is_mislabeled', 'rule_pred_mislabeled']].merge(
    llm_df[['id', 'llm_pred_mislabeled', 'raw_response']],
    on='id'
)

print(f"Merged rows: {len(merged)}")

Merged rows: 500


In [4]:
# ---- 3. Categorize ----
def categorize(row):
    rule_correct = row['rule_pred_mislabeled'] == row['is_mislabeled']
    llm_correct = row['llm_pred_mislabeled'] == row['is_mislabeled']

    if rule_correct and llm_correct:
        return 'both_correct'
    elif not rule_correct and not llm_correct:
        return 'both_wrong'
    elif llm_correct and not rule_correct:
        return 'llm_right_rule_wrong'
    else:
        return 'rule_right_llm_wrong'

merged['category'] = merged.apply(categorize, axis=1)

print("\nCategory counts:")
print(merged['category'].value_counts())


Category counts:
category
both_correct            272
llm_right_rule_wrong    146
both_wrong               43
rule_right_llm_wrong     39
Name: count, dtype: int64


In [7]:
# ---- 4. Pull disagreement examples ----
pd.set_option('display.max_colwidth', 60)

llm_right = merged[merged['category'] == 'llm_right_rule_wrong']
rule_right = merged[merged['category'] == 'rule_right_llm_wrong']
both_wrong = merged[merged['category'] == 'both_wrong']

print(f"\n--- LLM right, rule-based wrong ({len(llm_right)} cases) ---")
print(llm_right[['title', 'manufacturer_shown', 'true_manufacturer', 'is_mislabeled']].head(5))

print(f"\n--- Rule-based right, LLM wrong ({len(rule_right)} cases) ---")
print(rule_right[['title', 'manufacturer_shown', 'true_manufacturer', 'is_mislabeled']].head(5))

print(f"\n--- Both wrong ({len(both_wrong)} cases) ---")
print(both_wrong[['title', 'manufacturer_shown', 'true_manufacturer', 'is_mislabeled']].head(5))


--- LLM right, rule-based wrong (146 cases) ---
                                                      title  \
0                                    zonealarm anti-spyware   
3    zonealarm firewall pro 5 (free upgrade to new version)   
4                         the guild 2 - pirates of the seas   
6   symc backup exec aofo 11d win advanced open file option   
19                           imsi resume writer deluxe [lb]   

          manufacturer_shown         true_manufacturer  is_mislabeled  
0                  zone labs                 zone labs              0  
3                  zone labs                 zone labs              0  
4   dreamcatcher interactive  dreamcatcher interactive              0  
6                   symantec                  symantec              0  
19               imsi design               imsi design              0  

--- Rule-based right, LLM wrong (39 cases) ---
                                        title       manufacturer_shown  \
5                 

In [6]:
# ---- 5. Save for report use ----
llm_right.to_csv('../data/processed/brand_disagree_llm_right.csv', index=False)
rule_right.to_csv('../data/processed/brand_disagree_rule_right.csv', index=False)
both_wrong.to_csv('../data/processed/brand_both_wrong.csv', index=False)
print("\nSaved disagreement files to data/processed/")




Saved disagreement files to data/processed/
